In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit

In [2]:
rng = np.random.default_rng(42)

In [3]:
def make_search_data(n_queries=1200):
    frames = []
    for q in range(n_queries):
        n = rng.integers(8, 41)                       # retrieval returns 8-40 candidates
        difficulty  = rng.uniform(0.5, 1.5)           # some queries are just harder
        text_match  = rng.gamma(2.0, 1.0, n) * difficulty     # BM25-like lexical score
        semantic    = 0.7 * text_match + rng.normal(0, 0.8, n)  # embedding sim (correlated!)
        rating      = np.clip(rng.normal(4.0, 0.6, n), 1, 5)    # product stars
        log_reviews = rng.normal(4.0, 1.5, n)                   # log(1+#reviews): popularity
        price_ratio = rng.lognormal(0, 0.4, n)                  # price / category median
        freshness   = rng.exponential(180, n)                   # days since listed
        in_stock    = (rng.uniform(0, 1, n) < 0.9).astype(float)
        noise_feat  = rng.normal(0, 1, n)                       # pure noise (sanity feature)

        # latent utility: nonlinear price effect + interaction + irreducible noise
        u = (2.0 * text_match + 0.8 * semantic
             + 0.5 * (rating - 3) + 0.25 * log_reviews
             - 1.2 * np.abs(np.log(price_ratio))      # V-shape: too cheap OR too dear hurts
             - 0.002 * freshness
             + 1.5 * in_stock + 0.5 * text_match * in_stock   # relevance only pays if buyable
             + rng.normal(0, 1.0, n))                 # label noise -> NDCG ceiling < 1
        frames.append(pd.DataFrame(dict(
            qid=q, text_match=text_match, semantic=semantic, rating=rating,
            log_reviews=log_reviews, price_ratio=price_ratio, freshness=freshness,
            in_stock=in_stock, noise_feat=noise_feat, u=u)))
    df = pd.concat(frames, ignore_index=True)
    # global thresholds -> skewed grades: ~55% zeros ... ~3% grade-4 ("perfect hit")
    cuts = np.quantile(df.u, [0.55, 0.78, 0.90, 0.97])
    df["rel"] = np.digitize(df.u, cuts)               # 0..4  (<= 31 required: exp gain!)
    return df.drop(columns="u")

In [4]:
df = make_search_data()
FEATURES = [c for c in df.columns if c not in ("qid", "rel")]

In [5]:
df.head(23)

,qid,text_match,semantic,rating,log_reviews,price_ratio,freshness,in_stock,noise_feat,rel
0,0,2.662045,1.520769,4.369588,4.175029,0.828491,187.828467,1.0,-1.195840,1
1,0,1.724922,0.925739,4.677383,4.328033,0.774490,106.654356,1.0,0.486972,0
2,0,1.544521,1.507012,3.931632,5.307143,0.895783,8.001640,1.0,-0.469402,0
3,0,2.891046,2.316088,3.495906,4.335393,1.818436,172.188502,1.0,0.012494,1
4,0,1.646205,1.482529,3.505311,5.018370,0.707277,111.848956,1.0,0.480747,1
5,0,2.202614,1.886486,4.390356,4.101369,1.473015,271.433048,1.0,0.446531,1
6,0,2.055661,3.152281,4.445953,4.433679,0.510100,365.040285,1.0,0.665385,1
7,0,2.889317,1.697390,4.325893,4.946932,0.874630,59.772387,1.0,-0.098485,2
8,0,1.351253,0.536083,3.600694,1.814266,1.067267,8.955130,1.0,-0.423298,0
9,0,3.563592,1.843496,4.139297,3.520493,1.264262,166.468002,1.0,-0.079718,2


In [6]:
# ============================================================
# 2. GROUP-AWARE split: queries, not rows, go to train/val/test.
#    Splitting one query's docs across sets = leakage (shared
#    query context) and makes per-query NDCG meaningless.
# ============================================================
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=0)
trval_idx, test_idx = next(gss.split(df, groups=df.qid))
trval, test = df.iloc[trval_idx], df.iloc[test_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=1)
tr_idx, val_idx = next(gss2.split(trval, groups=trval.qid))
train, val = trval.iloc[tr_idx], trval.iloc[val_idx]

# XGBoost REQUIRES samples sorted by qid (non-decreasing)
train = train.sort_values("qid"); val = val.sort_values("qid"); test = test.sort_values("qid")

In [7]:
# ============================================================
# 3. Train the LambdaMART ranker (sklearn interface)
# ============================================================
ranker = xgb.XGBRanker(
    objective="rank:ndcg",                 # LambdaMART with |ΔNDCG| pair weights
    eval_metric=["ndcg@5", "ndcg@10"],     # LAST metric drives early stopping
    lambdarank_pair_method="topk",         # default since 2.0: focus pairs on list head
    lambdarank_num_pair_per_sample=10,     # topk => truncation level: optimize ~NDCG@10
    n_estimators=1500, learning_rate=0.05,
    max_depth=6, min_child_weight=10,      # a leaf must cover enough pair-mass
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", early_stopping_rounds=50, random_state=42)

ranker.fit(train[FEATURES], train.rel,
           qid=train.qid,                              # group structure for pair building
           eval_set=[(val[FEATURES], val.rel)],
           eval_qid=[val.qid],                         # one qid array per eval_set entry
           verbose=25)
print("best_iteration:", ranker.best_iteration, "| best val ndcg@10:", ranker.best_score)

[0]	validation_0-ndcg@5:0.82707	validation_0-ndcg@10:0.85880
[25]	validation_0-ndcg@5:0.97415	validation_0-ndcg@10:0.97653
[50]	validation_0-ndcg@5:0.97262	validation_0-ndcg@10:0.97506
[75]	validation_0-ndcg@5:0.97573	validation_0-ndcg@10:0.97916
[100]	validation_0-ndcg@5:0.97441	validation_0-ndcg@10:0.97879
[125]	validation_0-ndcg@5:0.97521	validation_0-ndcg@10:0.97870
[149]	validation_0-ndcg@5:0.97472	validation_0-ndcg@10:0.97842
best_iteration: 99 | best val ndcg@10: 0.9791935548249527


In [8]:
# ============================================================
# 4. Evaluate: mean per-query NDCG@10, vs. meaningful baselines
#    (implementing the metric ourselves = the math in code)
# ============================================================
def dcg_at_k(rels, k):
    rels = np.asarray(rels, dtype=float)[:k]
    return np.sum((2.0 ** rels - 1) / np.log2(np.arange(2, rels.size + 2)))

def ndcg_at_k(y_true, scores, k=10):
    order = np.argsort(-scores)                        # rank by score, descending
    idcg  = dcg_at_k(np.sort(y_true)[::-1], k)
    return 1.0 if idcg == 0 else dcg_at_k(np.asarray(y_true)[order], k) / idcg
    # idcg==0 -> 1.0 mirrors XGBoost's default; use metric "ndcg-" to count such lists as 0

def mean_ndcg(frame, score_col, k=10):
    return frame.groupby("qid").apply(
        lambda g: ndcg_at_k(g.rel.values, g[score_col].values, k)).mean()

test = test.copy()
test["s_model"]  = ranker.predict(test[FEATURES])      # arbitrary real scores; order is all
test["s_random"] = rng.normal(size=len(test))          # baseline 1: shuffle
test["s_pop"]    = test.log_reviews                    # baseline 2: sort by popularity
test["s_text"]   = test.text_match                     # baseline 3: sort by lexical match

for name in ["s_random", "s_pop", "s_text", "s_model"]:
    print(f"{name:9s}  NDCG@10 = {mean_ndcg(test, name):.4f}")

s_random   NDCG@10 = 0.3603
s_pop      NDCG@10 = 0.3797
s_text     NDCG@10 = 0.9474
s_model    NDCG@10 = 0.9690


In [9]:
# ============================================================
# 5. Serving: rank one query's candidates
# ============================================================
g = test[test.qid == test.qid.iloc[0]]
ranked = g.assign(score=ranker.predict(g[FEATURES])).sort_values("score", ascending=False)
print(ranked[["score", "rel", "text_match", "price_ratio", "in_stock"]].head(5))

       score  rel  text_match  price_ratio  in_stock
38  2.532403    4    5.887253     1.140194       1.0
19  1.214379    2    3.901119     1.211740       1.0
40  0.242267    2    3.020670     0.541428       1.0
39  0.184195    1    2.848482     1.995591       1.0
28 -0.280037    2    1.933584     0.959450       1.0


In [10]:
# ============================================================
# 6. Native API equivalent (qid goes straight into DMatrix)
# ============================================================
dtrain = xgb.DMatrix(train[FEATURES], label=train.rel, qid=train.qid.values)
dval   = xgb.DMatrix(val[FEATURES],   label=val.rel,   qid=val.qid.values)
params = dict(objective="rank:ndcg", eval_metric=["ndcg@10"],
              eta=0.05, max_depth=6, min_child_weight=10,
              lambdarank_pair_method="topk", lambdarank_num_pair_per_sample=10,
              tree_method="hist", seed=42)
booster = xgb.train(params, dtrain, num_boost_round=1500,
                    evals=[(dval, "valid")], early_stopping_rounds=50, verbose_eval=25)

[0]	valid-ndcg@10:0.95018


[25]	valid-ndcg@10:0.97451
[50]	valid-ndcg@10:0.97661
[75]	valid-ndcg@10:0.97814
[100]	valid-ndcg@10:0.97746
[121]	valid-ndcg@10:0.97644
